1 Tạo socket



In [3]:
import socket
import threading
HOST = '0.0.0.0'
PORT = 5000
server = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
server.bind((HOST, PORT))
server.listen()
server.settimeout(0.5)
clients = {}
clients_lock = threading.Lock()
workers = []
stop = threading.Event()
print('Server:', server.getsockname())


Server: ('0.0.0.0', 5000)


2 Nhận client và chuyển tiếp tin



In [4]:
def handle_client(conn, address):
    try:
        with conn.makefile('rb') as reader:
            while raw := reader.readline(4097):
                if len(raw) > 4096 or not raw.endswith(b'\n'):
                    break
                
                msg = f'[{address[1]}] {raw.decode().rstrip()}\n'.encode()
                print(msg.decode().rstrip())
                
                with clients_lock:
                    targets = list(clients.items())
                for target, lock in targets:
                    try:
                        with lock:
                            target.sendall(msg)
                    except OSError:
                        pass
    except (OSError, UnicodeError):
        pass
    finally:
        with clients_lock:
            clients.pop(conn, None)
        conn.close()

def accept_clients():
    while not stop.is_set():
        try:
            conn, addr = server.accept()
            with clients_lock:
                clients[conn] = threading.Lock()
            threading.Thread(target=handle_client, args=(conn, addr), daemon=True).start()
        except socket.timeout:
            continue
        except OSError:
            break

threading.Thread(target=accept_clients, daemon=True).start()
print('Đang chờ client. Bây giờ mở notebook Client A và Client B.')

Đang chờ client. Bây giờ mở notebook Client A và Client B.


3 Dừng server


In [5]:
stop.set()
server.close()

with clients_lock:
    for conn in list(clients):
        try:
            conn.shutdown(socket.SHUT_RDWR)
        except OSError:
            pass
        conn.close()

print('Đã dừng server')


Đã dừng server
